# Parameters: settings and records

`ariel.parameters` is where ARIEL keeps its fixed numbers and the base classes of its records. There are three kinds of things in it: **settings** (`ArielModulesConfig`: masses and sizes of the robot modules and the servo of the hinge, `MujocoConfig`, `ArielConfig`) that can be overridden from environment variables; **record bases** (`FrozenModel`, `MutableModel`) that every pydantic record in ARIEL derives from; and **type aliases** (`Position`, `FloatArray`, ...).

In this notebook you read the module settings, override one with an environment variable, see how the two record bases treat changes and unknown fields, and check the array types. No graphics backend is needed.

In [1]:
# Standard library
import os

In [2]:
# Third-party libraries
import astropy.units as u
import numpy as np
import pydantic
from rich.table import Table

In [3]:
# Local libraries
from ariel import console
from ariel.parameters import (
    ND_FLOAT_PRECISION,
    ND_INT_PRECISION,
    ArielConfig,
    ArielModulesConfig,
    FloatArray,
    FrozenModel,
    IntArray,
    MujocoConfig,
    MutableModel,
)

Nothing here is random, so there is no seed.

## The module settings

`ArielModulesConfig` holds what every module is made of, as measured on the printed parts: the size and mass of a brick and of the core, the hinge's parts (a fork on the parent, a cradle holding the servo on the child, screws and the connector), and the servo itself (a DFRobot DSS-M15). Every value is an astropy quantity, written in the unit it was measured in (grams, millimetres, kg·cm, µs) and stored in SI, so the table below shows kilograms, metres, newton metres and radians per second. The hinge's child side gets heavier the further the hinge is from the core, because the servo's cable runs back to the core.

In [4]:
modules = ArielModulesConfig()
table = Table(title="ArielModulesConfig")
table.add_column("setting")
table.add_column("value")
for part in ("brick", "core", "hinge", "servo"):
    for name, value in getattr(modules, part):
        if isinstance(value, pydantic.BaseModel):
            for inner, number in value:
                table.add_row(f"{part}.{name}.{inner}", f"{number:.4g}")
        elif isinstance(value, tuple):
            sides = ", ".join(f"{side:.4g}" for side in value)
            table.add_row(f"{part}.{name}", sides)
        else:
            table.add_row(f"{part}.{name}", f"{value:.4g}")
table.add_row("hinge parent side", f"{modules.hinge.parent_mass:.4g}")
for steps in (1, 4, 12):
    child = modules.hinge.child_mass(steps)
    table.add_row(
        f"hinge child side, {steps} steps from the core", f"{child:.4g}"
    )
console.print(table)

                           ArielModulesConfig                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ setting                                  ┃ value                      ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ brick.mass                               │ 0.05953 kg                 │
│ brick.size                               │ 0.075 m, 0.075 m, 0.075 m  │
│ core.mass                                │ 1.386 kg                   │
│ core.size                                │ 0.15 m, 0.15 m, 0.15 m     │
│ hinge.length                             │ 0.075 m                    │
│ hinge.width                              │ 0.052 m                    │
│ hinge.axis_from_parent                   │ 0.03174 m                  │
│ hinge.fork_mass                          │ 0.01148 kg                 │
│ hinge.cradle_mass                        │ 0.01649 kg                 │
│ hinge.plate_screws_mass                  │ 0.004165 kg                │
│ hinge.horn_screws_mass                   │ 0.00056 kg                 │
│ hinge.connector_mass                     │ 0.00221 kg                 │
│ hinge.bare_servo_mass                    │ 0.05724 kg                 │
│ hinge.cable_mass_per_length              │ 0.01135 kg / m             │
│ hinge.route_per_step                     │ 0.075 m                    │
│ servo.min_voltage                        │ 4.8 V                      │
│ servo.max_voltage                        │ 7.2 V                      │
│ servo.at_6_volts.voltage                 │ 6 V                        │
│ servo.at_6_volts.speed                   │ 5.818 rad / s              │
│ servo.at_6_volts.stall_torque            │ 1.324 N m                  │
│ servo.at_6_volts.stall_current           │ 1.8 A                      │
│ servo.at_6_volts.quiescent_current       │ 0.08 A                     │
│ servo.at_6_volts.standby_current         │ 0.004 A                    │
│ servo.at_7_2_volts.voltage               │ 7.2 V                      │
│ servo.at_7_2_volts.speed                 │ 6.545 rad / s              │
│ servo.at_7_2_volts.stall_torque          │ 1.471 N m                  │
│ servo.at_7_2_volts.stall_current         │ 2 A                        │
│ servo.at_7_2_volts.quiescent_current     │ 0.1 A                      │
│ servo.at_7_2_volts.standby_current       │ 0.005 A                    │
│ servo.angle_range                        │ 3.142 rad                  │
│ servo.gear_ratio                         │ 310                        │
│ servo.size                               │ 0.0545 m, 0.02 m, 0.0475 m │
│ servo.wire_length                        │ 0.32 m                     │
│ servo.horn_spline_teeth                  │ 25                         │
│ servo.horn_spline_diameter               │ 0.0058 m                   │
│ servo.min_pulse_width                    │ 0.0005 s                   │
│ servo.max_pulse_width                    │ 0.0025 s                   │
│ servo.neutral_pulse_width                │ 0.0015 s                   │
│ servo.min_pwm_frequency                  │ 50 Hz                      │
│ servo.max_pwm_frequency                  │ 330 Hz                     │
│ hinge parent side                        │ 0.01621 kg                 │
│ hinge child side, 1 steps from the core  │ 0.08095 kg                 │
│ hinge child side, 4 steps from the core  │ 0.0835 kg                  │
│ hinge child side, 12 steps from the core │ 0.09032 kg                 │
└──────────────────────────────────────────┴────────────────────────────┘

In [5]:
servo = modules.servo
assert modules.brick.mass.unit == u.kg
assert all(side.unit == u.m for side in modules.brick.size)
assert modules.hinge.child_mass(12) > modules.hinge.child_mass(1)
assert servo.angle_range.unit == u.rad
assert servo.at_6_volts.stall_torque.unit == u.N * u.m
assert servo.at_7_2_volts.speed > servo.at_6_volts.speed
assert servo.min_pwm_frequency < servo.max_pwm_frequency
assert MujocoConfig().floor_name == "floor"

## How the simulated hinge uses its servo

`servo_settings` (`ServoSettings`, goal servo-dynamics) chooses how the simulated hinge uses the DSS-M15: the supply voltage (the datasheet rates 6 V and 7.2 V), the angle error at which the servo pushes its full torque, and the motor's rotor inertia, which the datasheet does not give, so it is an estimate from a comparable maxon motor. From these and the datasheet it derives the values MuJoCo needs: the torque cap, the damping that makes the motor's torque-speed line (full torque at rest, none at the no-load speed), the position gain, and the armature (the rotor inertia as the joint feels it through the 310:1 gear).

In [6]:
settings = modules.servo_settings
derived = Table(title="ServoSettings, derived for the DSS-M15")
derived.add_column("value")
derived.add_column("SI")
for name in (
    "stall_torque",
    "no_load_speed",
    "damping",
    "stiffness",
    "armature",
):
    derived.add_row(name, f"{getattr(settings, name)(servo):.4g}")
console.print(
    f"voltage {settings.voltage:g}, holding error {settings.holding_error:.4g}"
)
console.print(derived)
console.print(f"rotor inertia: {settings.rotor_inertia_source}")

voltage 6 V, holding error 0.08727 rad

ServoSettings, derived for the DSS-M15
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ value         ┃ SI                 ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ stall_torque  │ 1.324 N m          │
│ no_load_speed │ 5.818 rad / s      │
│ damping       │ 0.2276 N m s / rad │
│ stiffness     │ 15.17 N m / rad    │
│ armature      │ 0.007842 m2 kg     │
└───────────────┴────────────────────┘

rotor inertia: estimate: maxon A-max 16, part 110079 (0.816 g cm2; stall torque 4.49 mN m, 1.39 N m after the 310:1
gear, close to the DSS-M15's 1.32 N m); the datasheet gives none

In [7]:
stall = settings.stall_torque(servo)
assert u.isclose(settings.damping(servo) * settings.no_load_speed(servo), stall)
assert u.isclose(settings.stiffness(servo) * settings.holding_error, stall)
assert settings.armature(servo).unit == u.kg * u.m**2

## Override a setting from the environment

These classes are `pydantic-settings` classes, so a setting is read from an environment variable. A nested value is named by its path, with `__` between the parts, and it needs its unit: `BRICK__MASS="0.1 kg"`. A bare number such as `"0.1"` is refused, so a value in the wrong unit cannot slip in. The next cell sets `BRICK__MASS` and `VERBOSE`, builds the configs, and removes the variables again (in `finally`, so a failure cannot leave them behind); then it shows the refusal.

In [8]:
os.environ["BRICK__MASS"] = "0.1 kg"
os.environ["VERBOSE"] = "false"
try:
    heavy = ArielModulesConfig()
    quiet = ArielConfig()
finally:
    del os.environ["BRICK__MASS"]
    del os.environ["VERBOSE"]
console.print(
    f"brick mass {modules.brick.mass} -> {heavy.brick.mass}; "
    f"verbose {ArielConfig().verbose} -> {quiet.verbose}"
)

os.environ["BRICK__MASS"] = "0.1"
try:
    ArielModulesConfig()
except pydantic.ValidationError as error:
    refusal = error.errors()[0]["msg"]
finally:
    del os.environ["BRICK__MASS"]
console.print(f"BRICK__MASS=0.1 -> {refusal}")

brick mass 0.059528 kg -> 0.1 kg; verbose True -> False

BRICK__MASS=0.1 -> Value error, needs the dimension of kg, got no unit

In [9]:
assert heavy.brick.mass > modules.brick.mass
assert "needs the dimension of kg" in refusal
assert not quiet.verbose
assert ArielConfig().verbose  # the variable is gone again

## The two record bases

A `FrozenModel` cannot change after it is created. A `MutableModel` can. Both reject a field they do not know, which catches a misspelt name instead of silently ignoring it.

In [10]:
class Frozen(FrozenModel):
    size: int


class Mutable(MutableModel):
    size: int


NEW_SIZE = 2
frozen, mutable = Frozen(size=1), Mutable(size=1)
mutable.size = NEW_SIZE
try:
    field = "size"  # set by name: the type checkers know the field is frozen
    setattr(frozen, field, NEW_SIZE)
except pydantic.ValidationError as error:
    console.print(f"frozen: {error.errors()[0]['type']}")
try:
    Mutable.model_validate({"size": 1, "sise": 2})
except pydantic.ValidationError as error:
    console.print(f"unknown field: {error.errors()[0]['type']}")

frozen: frozen_instance

unknown field: extra_forbidden

In [11]:
assert mutable.size == NEW_SIZE
assert frozen.size == 1
rejected = []
for model in (Frozen, Mutable):
    try:
        model.model_validate({"size": 1, "sise": 2})
    except pydantic.ValidationError:
        rejected.append(model.__name__)
assert rejected == ["Frozen", "Mutable"]

## Array types

`FloatArray` and `IntArray` are the aliases ARIEL uses in type hints, with fixed precision: 64-bit floats and 32-bit integers. The constants say which dtype to ask numpy for.

In [12]:
positions: FloatArray = np.zeros((4, 3), dtype=ND_FLOAT_PRECISION)
indices: IntArray = np.arange(4, dtype=ND_INT_PRECISION)
console.print(f"positions {positions.dtype}, indices {indices.dtype}")
assert positions.dtype == np.float64
assert indices.dtype == np.int32

positions float64, indices int32